# Step 1 (Kaggle): Multi-basin LSTM vs TFT on CAMELS-IND

Run with **Save Version → Save & Run All (Commit)**. It keeps running on Kaggle after you close the browser. Results appear in the notebook's **Output** tab.

Settings needed (right panel): **Accelerator: GPU T4 x2** and **Internet: On**.

In [ ]:
import subprocess, sys, os

LOG = '/kaggle/working/results/run_log.txt'
os.makedirs('/kaggle/working/results', exist_ok=True)

def run(cmd):
    """Run a shell command, show its output live, save it to the log, and stop if it fails."""
    print(f'\n$ {cmd}', flush=True)
    with open(LOG, 'a') as log:
        log.write(f'\n$ {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Step failed: {cmd} (see the output above or results/run_log.txt)')

run('nvidia-smi -L || echo no GPU')
run('cd /kaggle/working && (test -d flood || git clone -q https://github.com/yatharth7115/flood.git) && cd flood && git pull -q && git log --oneline -1')
os.chdir('/kaggle/working/flood')
run('pip install -q pytorch-forecasting pyyaml tabulate')

## Settings
`QUICK_RUN = True` uses 40 gauges (about 30 min). `False` uses all gauges (about 1.5–2.5 hours).

In [ ]:
import yaml
QUICK_RUN = False

cfg = yaml.safe_load(open('configs/step1.yaml'))
cfg['data']['data_dir'] = '/tmp/camels_data'                   # data is not kept in the output
cfg['data']['processed_file'] = '/tmp/camels_data/processed.parquet'
cfg['results_dir'] = '/kaggle/working/results'                  # shows up in the Output tab
cfg['data']['max_gauges'] = 40 if QUICK_RUN else None
if QUICK_RUN:
    cfg['lstm']['epochs'] = 10
    cfg['tft']['epochs'] = 8
os.makedirs('/tmp/camels_data', exist_ok=True)
yaml.safe_dump(cfg, open('configs/run.yaml', 'w'))
print(yaml.safe_dump(cfg['data']))

## 1. Download and inspect CAMELS-IND

In [ ]:
run('python src/download.py configs/run.yaml')
run('python src/explore.py /tmp/camels_data')

## 2. Build the clean dataset

In [ ]:
run('python src/dataset.py configs/run.yaml')

## 3. Train the LSTM baseline

In [ ]:
run('python src/train_lstm.py configs/run.yaml')

## 4. Train the Temporal Fusion Transformer

In [ ]:
run('python src/train_tft.py configs/run.yaml')

## 5. Compare on the test period (2016–2020)

In [ ]:
run('python src/evaluate.py configs/run.yaml')
from IPython.display import Image, display, Markdown
R = cfg['results_dir']
display(Markdown(open(f'{R}/summary.md').read()))
for f in ['nse_cdf.png', 'hydrograph_focus.png', 'tft_variable_importance.png']:
    if os.path.exists(f'{R}/{f}'):
        display(Image(f'{R}/{f}', width=700))
run(f'rm -rf {R}/tft_ckpt')   # keep the output small